# EV Charging Readiness & Accessibility

This notebook is the auditable preprocessing record for the Power BI project. It loads the downloaded NREL station inventory and Census population estimates, cleans the fields, creates station/state metrics, exports the dashboard tables, and runs validation checks.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'scripts').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURES_DIR = PROJECT_ROOT / 'outputs' / 'figures'
for directory in (PROCESSED_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

from scripts.prepare_data import (
    create_dimensions,
    create_encoded_features,
    create_group_summary,
    create_state_summary,
    create_year_summary,
    load_and_clean_stations,
    load_population,
    make_figures,
    write_data_dictionary,
)

raw_station_path = RAW_DIR / 'nrel_ev_stations.json'
raw_population_path = RAW_DIR / 'census_state_population_2025.csv'
if not raw_station_path.exists() or not raw_population_path.exists():
    from scripts.download_data import main as download_sources
    download_sources()

print(f'Project root: {PROJECT_ROOT}')
print(f'Python: {sys.executable}')

Project root: C:\Users\anude\OneDrive\Documents\acheivments\ev-charging-readiness
Python: C:\Users\anude\OneDrive\Documents\acheivments\ev-charging-readiness\.venv\Scripts\python.exe


In [2]:
raw_payload = json.loads(raw_station_path.read_text(encoding='utf-8'))
raw_stations = pd.DataFrame(raw_payload['fuel_stations'])
raw_population = pd.read_csv(raw_population_path, dtype={'STATE': 'string'})

print(f'Raw NREL records: {len(raw_stations):,}')
print(f'Raw Census rows: {len(raw_population):,}')
print(f"Fuel types: {raw_stations['fuel_type_code'].value_counts().to_dict()}")
raw_stations[['id', 'station_name', 'state', 'status_code', 'access_code', 'ev_network']].head(3)

Raw NREL records: 89,394
Raw Census rows: 66
Fuel types: {'ELEC': 89394}


,id,station_name,state,status_code,access_code,ev_network
0,1517,LADWP - Truesdale Center,CA,E,private,SHELL_RECHARGE
1,1523,Los Angeles Convention Center,CA,E,public,Non-Networked
2,1525,LADWP - John Ferraro Building,CA,E,private,Non-Networked


In [3]:
raw_profile = pd.DataFrame({
    'metric': ['rows', 'columns', 'missing latitude', 'missing longitude', 'missing status'],
    'value': [
        len(raw_stations),
        len(raw_stations.columns),
        raw_stations['latitude'].isna().mean(),
        raw_stations['longitude'].isna().mean(),
        raw_stations['status_code'].isna().mean(),
    ],
})
raw_profile

,metric,value
0,rows,89394.0
1,columns,76.0
2,missing latitude,0.0
3,missing longitude,0.0
4,missing status,0.0


## Clean the station inventory

The cleaning function normalizes identifiers, dates, access/status labels, charging-port counts, network/facility text, and the transparent 0–100 readiness heuristic.

A separate analysis matrix adds missing-value indicators, median-fills numeric features, and one-hot encodes low-cardinality labels with pandas.get_dummies.

In [4]:
stations, extract_date = load_and_clean_stations()
station_preview_columns = [
    'station_id', 'station_name', 'state_abbr', 'status_label',
    'access_label', 'total_ports', 'dc_fast_ports', 'readiness_score',
]
print(f'Clean station rows: {len(stations):,}')
print(f'Extract date: {extract_date}')
stations[station_preview_columns].head(10)

Clean station rows: 89,394
Extract date: 2026-09-20


,station_id,station_name,state_abbr,status_label,access_label,total_ports,dc_fast_ports,readiness_score
0,1517,LADWP - Truesdale Center,CA,Available,Private,59.0,2.0,65
1,1523,Los Angeles Convention Center,CA,Available,Public,8.0,0.0,75
2,1525,LADWP - John Ferraro Building,CA,Available,Private,394.0,15.0,65
3,1531,LADWP - Haynes Power Plant,CA,Available,Private,31.0,0.0,40
4,1552,LADWP - Harbor Generating Station,CA,Available,Private,12.0,0.0,40
5,1556,LADWP - Sylmar West,CA,Available,Private,9.0,0.0,40
6,1572,LADWP - EV Service Center,CA,Available,Private,44.0,1.0,65
7,1573,LADWP - Fairfax Center,CA,Available,Private,14.0,0.0,45
8,1589,LADWP - Palmetto Center,CA,Available,Private,36.0,1.0,65
9,6355,Scripps Green Hospital,CA,Available,Public,1.0,0.0,65


In [5]:
population = load_population()
print(f'Population states/territories retained: {len(population):,}')
population.head()

Population states/territories retained: 52


,state_abbr,state_name,population_2025
14,AL,Alabama,5193088
15,AK,Alaska,737270
16,AZ,Arizona,7623818
17,AR,Arkansas,3114791
18,CA,California,39355309


## Create Power BI summary tables

State metrics use 2025 population as the denominator. The priority score is 70% coverage gap plus 30% availability gap versus the median state.

In [6]:
state_summary = create_state_summary(stations, population)
network_summary = create_group_summary(stations, 'ev_network')
facility_summary = create_group_summary(stations, 'facility_type_label')
open_year_summary = create_year_summary(stations)
dimensions = create_dimensions(stations, population)

state_summary[['state_abbr', 'population_2025', 'public_available_per_100k', 'availability_rate', 'priority_score', 'priority_band']].head(10)

,state_abbr,population_2025,public_available_per_100k,availability_rate,priority_score,priority_band
0,PR,3184835,0.596577,0.760000,75.0,Priority review
1,LA,4618189,5.478338,0.955696,50.9,Priority review
2,MS,2954160,7.548677,0.968872,42.8,Watch list
3,KY,4606864,8.769523,0.941176,39.0,Watch list
4,OK,4123288,9.240199,0.961451,36.7,Watch list
5,AK,737270,9.630122,0.964286,35.1,Watch list
6,IN,6973333,10.239006,0.969962,32.7,Watch list
7,WV,1766147,10.248298,0.976190,32.5,Watch list
8,AL,5193088,10.687283,0.966472,31.1,Watch list
9,TX,31709821,11.8796,0.970225,26.6,Watch list


In [7]:
encoded_features = create_encoded_features(stations)
encoded_features.to_csv(PROCESSED_DIR / 'station_features_encoded.csv', index=False)
encoded_columns = encoded_features.columns[encoded_features.columns.str.contains('_label_|readiness_band_', regex=True)]
print(f'One-hot encoded analysis matrix: {encoded_features.shape[0]:,} rows x {encoded_features.shape[1]:,} columns')
print(f'Encoded categorical features: {len(encoded_columns):,}')
display(encoded_features.head(3))
print('Top networks')
display(network_summary.head(10))
print('Top facility types')
display(facility_summary.head(10))
print('Opening-year trend')
display(open_year_summary.tail(10))

One-hot encoded analysis matrix: 89,394 rows x 97 columns
Encoded categorical features: 71


,station_id,latitude_missing,latitude,longitude_missing,longitude,station_age_years_missing,station_age_years,level_1_ports_missing,level_1_ports,level_2_ports_missing,...,facility_type_label_Storage,facility_type_label_Street Parking,facility_type_label_Travel Center,facility_type_label_Truck Stop,facility_type_label_Unknown,facility_type_label_Utility,facility_type_label_Workplace,readiness_band_Partly ready,readiness_band_Priority review,readiness_band_Ready
0,1517,0,34.248319,0,-118.387971,0,27,0,0.0,0,...,0,0,0,0,0,1,0,1,0,0
1,1523,0,34.040539,0,-118.271387,0,31,0,0.0,0,...,0,0,0,0,0,0,0,1,0,0
2,1525,0,34.059133,0,-118.248589,0,27,0,0.0,0,...,0,0,0,0,0,1,0,1,0,0


Top networks


,ev_network,stations,available_stations,public_stations,dc_fast_stations,total_ports,mean_readiness_score,availability_rate,public_share
16,ChargePoint Network,48634,47997,48327,4576,87874.0,61.815602,0.986902,0.993688
61,Non-Networked,8830,8438,5110,701,33554.0,53.159117,0.955606,0.578709
7,Blink Network,6215,5895,6201,702,23068.0,66.990346,0.948512,0.997747
86,Tesla Destination,4512,4512,3917,0,18141.0,64.280807,1.000000,0.868129
85,Tesla,3122,3122,3122,3122,38407.0,95.203395,1.000000,1.000000
22,EV Connect,1606,1439,1517,821,5759.0,71.889788,0.896015,0.944583
3,AMPUP,1324,1324,832,7,5356.0,52.828550,1.000000,0.628399
40,FLO,1240,1219,1235,100,3096.0,63.762097,0.983065,0.995968
35,Electrify America,1217,1216,1217,1187,6157.0,93.833196,0.999178,1.000000
96,eVgo Network,1214,1181,1214,1209,5430.0,91.622735,0.972817,1.000000


Top facility types


,facility_type_label,stations,available_stations,public_stations,dc_fast_stations,total_ports,mean_readiness_score,availability_rate,public_share
60,Unknown,67167,65647,66531,10266,167812.0,64.620275,0.977370,0.990531
22,Hotel,2825,2793,2766,322,11264.0,69.437168,0.988673,0.979115
7,Car Dealer,2172,2125,1218,535,3869.0,55.819521,0.978361,0.560773
42,Public,1995,1943,1995,1183,8992.0,79.972431,0.973935,1.000000
14,Fed Gov,1647,1502,50,25,10610.0,32.826351,0.911961,0.030358
51,Shopping Center,1582,1566,1557,1407,18875.0,91.615044,0.989886,0.984197
32,Other,1334,1331,1255,150,10992.0,71.697901,0.997751,0.940780
31,Office Bldg,1242,1230,536,51,8051.0,50.305958,0.990338,0.431562
36,Parking Lot,1041,1019,950,96,4125.0,66.095101,0.978866,0.912584
18,Gas Station,837,819,836,808,7942.0,93.458781,0.978495,0.998805


Opening-year trend


,open_year,new_stations,public_stations,available_stations,dc_fast_stations
19,2017,1549,1422,1525,156
20,2018,1910,1743,1873,361
21,2019,2756,2376,2715,523
22,2020,5767,5349,5640,852
23,2021,13920,13109,13663,1314
24,2022,9917,9486,9685,1390
25,2023,12168,11787,11945,2469
26,2024,12645,12080,12398,3011
27,2025,13850,13013,13568,3310
28,2026,9012,8989,8773,2045


In [8]:
output_tables = {
    'ev_stations_clean.csv': stations,
    'station_features_encoded.csv': encoded_features,
    'state_summary.csv': state_summary,
    'network_summary.csv': network_summary,
    'facility_summary.csv': facility_summary,
    'open_year_summary.csv': open_year_summary,
    **{f'{name}.csv': frame for name, frame in dimensions.items()},
}
for filename, frame in output_tables.items():
    frame.to_csv(PROCESSED_DIR / filename, index=False)
write_data_dictionary()

quality_report = pd.DataFrame({
    'metric': [
        'extract_date', 'station_rows', 'state_rows', 'missing_latitude_pct',
        'missing_longitude_pct', 'missing_total_ports_pct', 'unknown_access_pct',
        'unknown_status_pct',
    ],
    'value': [
        extract_date, len(stations), len(state_summary),
        stations['latitude'].isna().mean(), stations['longitude'].isna().mean(),
        stations['total_ports'].eq(0).mean(), stations['access_label'].eq('Unknown').mean(),
        stations['status_label'].eq('Unknown').mean(),
    ],
})
quality_report.to_csv(PROCESSED_DIR / 'quality_report.csv', index=False)
quality_report

,metric,value
0,extract_date,2026-09-20
1,station_rows,89394
2,state_rows,52
3,missing_latitude_pct,0.0
4,missing_longitude_pct,0.0
5,missing_total_ports_pct,0.000157
6,unknown_access_pct,0.0
7,unknown_status_pct,0.0


In [9]:
make_figures(stations, state_summary, open_year_summary)
print(f'Wrote {len(output_tables):,} Power BI tables to {PROCESSED_DIR}')
print(f'Wrote EDA figures to {FIGURES_DIR}')

Wrote 10 Power BI tables to C:\Users\anude\OneDrive\Documents\acheivments\ev-charging-readiness\data\processed
Wrote EDA figures to C:\Users\anude\OneDrive\Documents\acheivments\ev-charging-readiness\outputs\figures


In [10]:
assert len(stations) >= 80_000
assert stations['station_id'].notna().all()
assert stations['station_id'].is_unique
assert state_summary['state_abbr'].nunique() >= 50
assert stations['readiness_score'].between(0, 100).all()
assert state_summary['priority_score'].between(0, 100).all()
assert (PROCESSED_DIR / 'ev_stations_clean.csv').exists()
assert (PROCESSED_DIR / 'station_features_encoded.csv').exists()
assert encoded_features.shape[0] == stations.shape[0]
assert encoded_features.filter(regex='^(access_label|status_label|facility_type_label|readiness_band)_').shape[1] > 0

print('All preprocessing validation checks passed.')
print(f"Public share: {stations['is_public'].mean():.1%}")
print(f"Availability rate: {stations['is_available'].mean():.1%}")
print(f"DC-fast station share: {stations['has_dc_fast'].mean():.1%}")

All preprocessing validation checks passed.
Public share: 93.8%
Availability rate: 97.7%
DC-fast station share: 18.2%


In [11]:
state_summary.nlargest(5, 'priority_score')[
    ['state_abbr', 'priority_score', 'public_available_per_100k', 'availability_rate']
]

,state_abbr,priority_score,public_available_per_100k,availability_rate
0,PR,75.0,0.596577,0.760000
1,LA,50.9,5.478338,0.955696
2,MS,42.8,7.548677,0.968872
3,KY,39.0,8.769523,0.941176
4,OK,36.7,9.240199,0.961451
